# CodeSnap - Jupyter Version
Package: com.appwhiz.codesnap
Run all cells

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import textwrap, json, os
LANGUAGES = {'Python': {'file': 'solution.py', 'comment': '#'}, 'C++': {'file': 'solution.cpp', 'comment': '//'}, 'C': {'file': 'solution.c', 'comment': '//'}, 'Java': {'file': 'Solution.java', 'comment': '//'}, 'JavaScript': {'file': 'solution.js', 'comment': '//'}, 'C#': {'file': 'Solution.cs', 'comment': '//'}, 'Go': {'file': 'main.go', 'comment': '//'}, 'Kotlin': {'file': 'Solution.kt', 'comment': '//'}, 'Rust': {'file': 'main.rs', 'comment': '//'}}

def translate(text, lang):
    t=text.lower(); c=LANGUAGES[lang]['comment']; head=f"{c} From: {text[:100]}\n{c} Lang: {lang}\n\n"
    def w(b): return head + textwrap.dedent(b).strip() + "\n"
    if 'fizzbuzz' in t and lang=='Python': return w('def fizzbuzz(n=100):\n for i in range(1,n+1):\n  print("FizzBuzz" if i%15==0 else "Fizz" if i%3==0 else "Buzz" if i%5==0 else i)\nfizzbuzz()')
    if 'prime' in t and lang=='Python': return w('def is_prime(n):\n if n<=1: return False\n if n<=3: return True\n if n%2==0 or n%3==0: return False\n i=5\n while i*i<=n:\n  if n%i==0 or n%(i+2)==0: return False\n  i+=6\n return True\nprint(is_prime(17))')
    if lang=='Python': return w(f"# {text}\ndef solve(d=[1,2,3,4,5]):\n return [x*2 for x in d]\nprint(solve())")
    return w(f"{c} {text} -> {lang}")


In [2]:
HISTORY_FILE='codesnap_history.json'
history=[]
if os.path.exists(HISTORY_FILE):
    try: history=json.load(open(HISTORY_FILE))
    except: history=[]
def save_history(): json.dump(history[:20], open(HISTORY_FILE,'w'))
input_box=widgets.Textarea(placeholder='e.g. check if prime...', description='Scratch Pad:', layout=widgets.Layout(width='100%', height='100px'))
translate_btn=widgets.Button(description='Translate', button_style='success')
clear_btn=widgets.Button(description='Clear', button_style='warning')
lang_tabs=widgets.Tab()
code_outputs={}
for lang in LANGUAGES:
    code_outputs[lang]=widgets.Textarea(value='Code here...', layout=widgets.Layout(width='100%', height='350px'))
lang_tabs.children=tuple(code_outputs.values())
for i,lang in enumerate(LANGUAGES): lang_tabs.set_title(i, lang)
history_box=widgets.VBox()
out=widgets.Output()

def refresh_history():
    items=[]
    if history:
        items.append(widgets.HTML('<b>Recent (Delete removes):</b>'))
        for idx,h in enumerate(history):
            b1=widgets.Button(description=h[:40], layout=widgets.Layout(width='65%'))
            b2=widgets.Button(description='Delete', button_style='danger', layout=widgets.Layout(width='25%'))
            def make_load(txt): return lambda x: (setattr(input_box,'value',txt), on_translate(None))
            def make_del(i): return lambda x: (history.pop(i), save_history(), refresh_history())
            b1.on_click(make_load(h)); b2.on_click(make_del(idx))
            items.append(widgets.HBox([b1,b2]))
    history_box.children=tuple(items)

def on_translate(b):
    text=input_box.value.strip()
    if not text: return
    if text not in history: history.insert(0,text); save_history()
    refresh_history()
    for lang,area in code_outputs.items(): area.value=translate(text, lang)
    with out: clear_output(); print(f'Translated to {len(LANGUAGES)} langs')
translate_btn.on_click(on_translate)
clear_btn.on_click(lambda b: setattr(input_box,'value',''))
refresh_history()
display(widgets.VBox([input_box, widgets.HBox([translate_btn, clear_btn]), history_box, lang_tabs, out]))


In [3]:
def save_run(b):
    idx=lang_tabs.selected_index; lang=list(LANGUAGES.keys())[idx]; code=code_outputs[lang].value; fname=LANGUAGES[lang]['file']; open(fname,'w').write(code); print(f'Saved {fname}')
    if lang=='Python':
        try: exec(code)
        except Exception as e: print(e)
btn=widgets.Button(description='Save & Run', button_style='primary'); btn.on_click(save_run); display(btn)


Button(button_style='primary', description='Save & Run', style=ButtonStyle())